# Assignment 2 – Task 2

## Introduction

This task focuses on building a recommendation system using Spark MLlib. The dataset contains information about users, games, and user interactions such as purchases and playtime. The goal is to build a collaborative filtering model that can recommend games to users based on their behaviour.

## Data Loading and Exploration

In this section, the dataset is loaded into a Spark DataFrame and explored to understand its structure. The dataset contains user interactions with games, including whether a user purchased or played a game and the number of hours played. The schema is inspected to confirm data types and identify key fields that will be used in building the recommendation system.

In [0]:
# Load the Steam dataset
df = spark.read.csv(
    "/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv",
    header=False,
    inferSchema=True
)

# Rename columns correctly (4 columns only)
df = df.toDF("user_id", "game", "action", "hours")

# Show sample
display(df)

# Check schema
df.printSchema()

user_id,game,action,hours
151603712,The Elder Scrolls V Skyrim,purchase,1.0
151603712,The Elder Scrolls V Skyrim,play,273.0
151603712,Fallout 4,purchase,1.0
151603712,Fallout 4,play,87.0
151603712,Spore,purchase,1.0
151603712,Spore,play,14.9
151603712,Fallout New Vegas,purchase,1.0
151603712,Fallout New Vegas,play,12.1
151603712,Left 4 Dead 2,purchase,1.0
151603712,Left 4 Dead 2,play,8.9


root
 |-- user_id: integer (nullable = true)
 |-- game: string (nullable = true)
 |-- action: string (nullable = true)
 |-- hours: double (nullable = true)



## Exploratory Data Analysis (EDA)

In this section, the dataset is explored to better understand user behaviour and data distribution. Basic statistics and counts are calculated to examine the scale of the dataset, including the number of users, games, and total interactions. The distribution of user actions and the most popular games are also analysed. This helps to prepare the data for building the recommendation model.

In [0]:
#  dataset statistics
total_records = df.count()
unique_users = df.select("user_id").distinct().count()
unique_games = df.select("game").distinct().count()

print("Total records:", total_records)
print("Unique users:", unique_users)
print("Unique games:", unique_games)

Total records: 200000
Unique users: 12393
Unique games: 5155


In [0]:
# Count actions (purchase vs play)
display(df.groupBy("action").count())

action,count
purchase,129511
play,70489


Databricks visualization. Run in Databricks to view.

In [0]:
# Top 10 most popular games
display(
    df.groupBy("game")
      .count()
      .orderBy("count", ascending=False)
      .limit(10)
)

game,count
Dota 2,9682
Team Fortress 2,4646
Counter-Strike Global Offensive,2789
Unturned,2632
Left 4 Dead 2,1752
Counter-Strike Source,1693
Counter-Strike,1424
Garry's Mod,1397
The Elder Scrolls V Skyrim,1394
Warframe,1271


Databricks visualization. Run in Databricks to view.

In [0]:
# Average hours played
df.selectExpr("avg(hours) as average_hours_played").show()

+--------------------+
|average_hours_played|
+--------------------+
|  17.874383999999914|
+--------------------+



### Explanation of EDA Results

The dataset contains 200,000 user interaction records, involving 12,393 unique users and 5,155 unique games. This confirms a moderately sized implicit feedback dataset suitable for collaborative filtering.

**Action distribution:** There are 129,511 purchase interactions and 70,489 play interactions. This means approximately 35% of games purchased by a user are also associated with a recorded play session, while the remaining 65% of records are purchase-only, the user bought the game but no playtime was logged.

**Popularity bias:** The top 10 most interacted-with games are heavily dominated by free-to-play titles: Dota 2 alone accounts for 9,682 interactions, followed by Team Fortress 2 (4,646) and Counter-Strike: Global Offensive (2,789). This extreme popularity concentration is a known challenge in recommender systems without careful handling, the model risks over-recommending a small number of dominant titles to all users.

**Average hours played:** The average hours value across all rows (including purchase rows which have hours=1) is approximately 17.87 hours. However, this figure is heavily skewed by the raw interaction values: a small number of users have played games for hundreds or thousands of hours, pulling the mean well above the median. This skewness justifies applying a log transformation to the hours values before model training.

## Data Preparation for ALS

Before training the recommendation model, the dataset must be prepared in a format suitable for Spark MLlib ALS. ALS requires numeric identifiers for both users and items. The user IDs are already numeric, but the game names must be converted into integer IDs. In addition, an interaction value must be defined so that user behaviour can be used as implicit feedback for the recommendation system.

In [0]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window

# Create a unique integer ID for each game directly in Databricks
games_df = (
    df.select("game")
      .distinct()
      .withColumn("game_id", dense_rank().over(Window.orderBy("game")))
)

display(games_df.limit(10))

game,game_id
007 Legends,1
0RBITALIS,2
1... 2... 3... KICK IT! (Drop That Beat Like an Ugly Baby),3
10 Second Ninja,4
"10,000,000",5
100% Orange Juice,6
1000 Amps,7
12 Labours of Hercules,8
12 Labours of Hercules II The Cretan Bull,9
12 Labours of Hercules III Girl Power,10


In [0]:
# Join the generated game IDs back to the main dataset
df_with_ids = df.join(games_df, on="game", how="inner")

display(df_with_ids.limit(10))
df_with_ids.printSchema()

game,user_id,action,hours,game_id
The Elder Scrolls V Skyrim,151603712,purchase,1.0,4365
The Elder Scrolls V Skyrim,151603712,play,273.0,4365
Fallout 4,151603712,purchase,1.0,1679
Fallout 4,151603712,play,87.0,1679
Spore,151603712,purchase,1.0,3998
Spore,151603712,play,14.9,3998
Fallout New Vegas,151603712,purchase,1.0,1680
Fallout New Vegas,151603712,play,12.1,1680
Left 4 Dead 2,151603712,purchase,1.0,2476
Left 4 Dead 2,151603712,play,8.9,2476


root
 |-- game: string (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- action: string (nullable = true)
 |-- hours: double (nullable = true)
 |-- game_id: integer (nullable = false)



### Explanation of Results

A unique integer ID was generated for each game and then joined back to the main dataset. This step is necessary because ALS cannot use text values as item identifiers. Creating the IDs directly in Databricks avoids the need for an external file and keeps the data preparation process fully reproducible.

This structured format ensures that both users and items are represented numerically, which is required for training the collaborative filtering model.

## Creating Interaction Values

To train the ALS model, a numeric interaction value is required. Since the dataset contains both purchase and play actions, a combined interaction measure is created. Purchase actions are assigned a value of 1, while play actions use the number of hours played. This approach captures both ownership and engagement.

In [0]:
from pyspark.sql.functions import when

df_model = df_with_ids.withColumn(
    "interaction",
    when(col("action") == "purchase", 1)
    .otherwise(col("hours"))
)

display(df_model.limit(10))

game,user_id,action,hours,game_id,interaction
The Elder Scrolls V Skyrim,151603712,purchase,1.0,4365,1.0
The Elder Scrolls V Skyrim,151603712,play,273.0,4365,273.0
Fallout 4,151603712,purchase,1.0,1679,1.0
Fallout 4,151603712,play,87.0,1679,87.0
Spore,151603712,purchase,1.0,3998,1.0
Spore,151603712,play,14.9,3998,14.9
Fallout New Vegas,151603712,purchase,1.0,1680,1.0
Fallout New Vegas,151603712,play,12.1,1680,12.1
Left 4 Dead 2,151603712,purchase,1.0,2476,1.0
Left 4 Dead 2,151603712,play,8.9,2476,8.9


### Explanation: Creating Interaction Values

An interaction column was created to quantify user engagement with each game. Purchase actions were assigned a value of 1 to represent ownership, while play actions use the raw number of hours played as a proxy for engagement strength.

This combined approach captures two dimensions of user behaviour: whether a user owns a game (purchase) and how much they engage with it (play hours). For example, user 151603712 has 273 hours on The Elder Scrolls V: Skyrim and 87 hours on Fallout 4, reflecting strong engagement, while the purchase-only entries have a value of 1.

This represents an implicit feedback approach, where user preferences are inferred from observed behaviour rather than explicit ratings. One limitation is that the hours values are highly skewed  a small number of users play for hundreds of hours while most play far less  which could influence the model's training by giving disproportionate weight to extreme values.

## Train-Test Split and Initial Model Evaluation

To evaluate the recommendation model properly, the dataset is split into training and test sets. The ALS model is trained on the training data and then used to generate predictions on the test data. Root Mean Squared Error (RMSE) is used as an evaluation metric to measure how closely the predicted interaction values match the observed values.

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# Keep only the columns needed for modelling
als_data = df_model.select("user_id", "game_id", "interaction")

# Split the data into training and test sets
train_data, test_data = als_data.randomSplit([0.8, 0.2], seed=42)

print("Training rows:", train_data.count())
print("Test rows:", test_data.count())

Training rows: 160047
Test rows: 39953


In [0]:
from pyspark.ml.recommendation import ALS

# Initial ALS model
als_1 = ALS(
    userCol="user_id",
    itemCol="game_id",
    ratingCol="interaction",
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    rank=10,
    maxIter=10,
    regParam=0.1
)

model_1 = als_1.fit(train_data)

# Generate predictions on test data
predictions_1 = model_1.transform(test_data)

display(predictions_1.limit(10))

user_id,game_id,interaction,prediction
5250,853,1.0,0.6884165
5250,1132,1.0,0.20595214
5250,1249,1.0,0.43326065
5250,2071,1.0,0.71985894
5250,2079,1.0,0.121573195
5250,3532,1.0,0.14524512
76767,228,0.8,0.48481792
76767,443,24.0,0.42192245
76767,741,15.9,0.81948334
76767,742,1.0,0.8162981


In [0]:
# Evaluate the first model using RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="interaction",
    predictionCol="prediction"
)

rmse_1 = evaluator.evaluate(predictions_1)
print("RMSE for Model 1:", rmse_1)

RMSE for Model 1: 115.88536796747813


### Explanation: Train/Test Split and Model 1

The dataset was split into 160,047 training rows and 39,953 test rows using an 80/20 random split with seed 42 to ensure reproducibility.

The first ALS model was configured with `rank=10`, `maxIter=10`, and `regParam=0.1`. The `implicitPrefs=True` setting is essential here because the interaction values represent inferred preferences (hours played / purchases) rather than explicit user ratings. `coldStartStrategy="drop"` removes test rows where a user or item was not seen during training, preventing NaN predictions from distorting the RMSE calculation. `nonnegative=True` constrains the latent factors to non-negative values, which is appropriate for implicit feedback data where higher values always mean stronger preference.

**Model 1 achieved an RMSE of 115.89** on the test set. This value represents the average prediction error in units of the interaction scale (hours for play rows, 1.0 for purchase rows). The high absolute value is largely driven by the skewed distribution of hours a small number of rows with very high hour counts (e.g. 273 hours) contribute disproportionately to the squared error. This is a known limitation of RMSE on implicit feedback data with raw interaction values.

## Second ALS Run and Model Comparison

To improve the recommendation system, a second ALS model is trained using different hyperparameter values. This allows the performance of different model settings to be compared. Comparing multiple runs helps to identify which configuration produces better results.

In [0]:
# Second ALS model with different hyperparameters
als_2 = ALS(
    userCol="user_id",
    itemCol="game_id",
    ratingCol="interaction",
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    rank=20,
    maxIter=15,
    regParam=0.05
)

model_2 = als_2.fit(train_data)

# Generate predictions for the second model
predictions_2 = model_2.transform(test_data)

display(predictions_2.limit(10))

user_id,game_id,interaction,prediction
5250,853,1.0,0.86694366
5250,1132,1.0,0.35295618
5250,1249,1.0,0.64423335
5250,2071,1.0,0.87044746
5250,2079,1.0,0.28073427
5250,3532,1.0,0.27276528
76767,228,0.8,0.2118078
76767,443,24.0,0.22478604
76767,741,15.9,0.8863756
76767,742,1.0,0.8798655


In [0]:
# Evaluate the second model
rmse_2 = evaluator.evaluate(predictions_2)
print("RMSE for Model 2:", rmse_2)

RMSE for Model 2: 115.87811132248879


### Explanation: Model 2 Results

The second ALS model used `rank=20`, `maxIter=15`, and `regParam=0.05` — a higher-complexity configuration with more latent factors, more training iterations, and weaker regularisation.

**Model 2 achieved an RMSE of 115.88**, marginally lower than Model 1's 115.89. The difference is only 0.007, which is negligible in practical terms. This suggests that simply increasing rank and reducing regularisation provides no meaningful benefit on this dataset with these interaction values. The near-identical performance indicates that both models have learned a similar underlying structure, and the bottleneck is likely the scale of the raw interaction values rather than model complexity.

## Third ALS Model

In this section, a third ALS model is trained using a different set of hyperparameters. The purpose of this model is to further explore how parameter changes affect the performance of the recommender system.

As in the previous models, the ALS algorithm is used because it is suitable for collaborative filtering and can handle large-scale user–item interaction data efficiently in Spark.

This model is trained on the same training dataset and evaluated on the same test dataset to ensure a fair comparison. After training, predictions are generated and evaluated using RMSE to measure the accuracy of the model.

In [0]:
# Model 3: lower rank with stronger regularisation — tests underfitting/overfitting boundary
als_3 = ALS(
    userCol="user_id",
    itemCol="game_id",
    ratingCol="interaction",
    implicitPrefs=True,
    coldStartStrategy="drop",
    nonnegative=True,
    rank=5,
    maxIter=10,
    regParam=0.5
)

model_3 = als_3.fit(train_data)
predictions_3 = model_3.transform(test_data)

rmse_3 = evaluator.evaluate(predictions_3)
print("RMSE for Model 3:", rmse_3)

RMSE for Model 3: 115.89032696765078


In [0]:
# Compare all models
print("Model 1 RMSE:", rmse_1)
print("Model 2 RMSE:", rmse_2)
print("Model 3 RMSE:", rmse_3)

# Find the best model
best_rmse = min(rmse_1, rmse_2, rmse_3)

if best_rmse == rmse_1:
    print("Model 1 performed best because it has the lowest RMSE.")
elif best_rmse == rmse_2:
    print("Model 2 performed best because it has the lowest RMSE.")
else:
    print("Model 3 performed best because it has the lowest RMSE.")

Model 1 RMSE: 115.88536796747813
Model 2 RMSE: 115.87811132248879
Model 3 RMSE: 115.89032696765078
Model 2 performed best because it has the lowest RMSE.


## Explanation of Model 3

The third ALS model was created using a different combination of hyperparameters to test whether prediction accuracy could be improved further. In this case, a lower rank (5) and stronger regularisation (0.5) were used to explore the effect of reducing model complexity and preventing overfitting.

The model was trained using the same user IDs, game IDs, and interaction values as the previous models. This ensures that all models are evaluated under the same conditions and can be fairly compared.

**Model 3 achieved an RMSE of 115.90.** This value is slightly higher than Model 2 (115.88) and very similar to Model 1 (115.89), indicating that the stronger regularisation and lower rank reduced the model’s ability to capture user preferences effectively.

This result suggests that Model 3 is slightly underfitting the data and is therefore less suitable for this recommendation task.

### Explanation: MLflow Experiment Tracking

MLflow was used to log the parameters and evaluation metrics for all ALS runs in a structured, reproducible way. Each run was recorded under the experiment `/Shared/BDTT_Assignment2_Experiments` with the following parameters logged: `rank`, `maxIter`, `regParam`, and `implicitPrefs`, along with the `rmse` metric.

Using MLflow provides several benefits over manual comparison: all runs are stored with timestamps and can be compared visually in the Databricks MLflow UI, parameters and metrics are linked automatically, and the experiment is reproducible by any team member with access to the workspace. This reflects industry-standard MLOps practice for managing machine learning experiments at scale.

In [0]:
import mlflow
import mlflow.spark

In [0]:
import mlflow

mlflow.set_experiment("/Shared/BDTT_Assignment2_Experiments")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/1281990939411497', creation_time=1776702622319, experiment_id='1281990939411497', last_update_time=1776709231016, lifecycle_stage='active', name='/Shared/BDTT_Assignment2_Experiments', tags={'mlflow.experiment.sourceName': '/Shared/BDTT_Assignment2_Experiments',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'Group-20',
 'mlflow.ownerId': '85670146021981'}>

In [0]:
# Log Model 1
with mlflow.start_run(run_name="ALS_Model_1"):
    mlflow.log_param("rank", 10)
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("regParam", 0.1)
    mlflow.log_param("implicitPrefs", True)
    mlflow.log_metric("rmse", rmse_1)

In [0]:
# Log Model 2
with mlflow.start_run(run_name="ALS_Model_2"):
    mlflow.log_param("rank", 20)
    mlflow.log_param("maxIter", 15)
    mlflow.log_param("regParam", 0.05)
    mlflow.log_param("implicitPrefs", True)
    mlflow.log_metric("rmse", rmse_2)

In [0]:
 # log Model 3
 with mlflow.start_run(run_name="ALS_Model_3"):
    mlflow.log_param("rank", 5)
    mlflow.log_param("maxIter", 10)
    mlflow.log_param("regParam", 0.5)
    mlflow.log_param("implicitPrefs", True)
    mlflow.log_metric("rmse", rmse_3)

### Explanation of Results

MLflow was used to record the parameter settings and RMSE values for both ALS runs. This improves experiment management and makes it easier to compare different models in a clear and reproducible way. Using MLflow also supports a more structured machine learning workflow.

In [0]:
comparison_data = [
    ("Model 1", 10, 10, 0.1,  rmse_1),
    ("Model 2", 20, 15, 0.05, rmse_2),
    ("Model 3", 5,  10, 0.5,  rmse_3),
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["model", "rank", "maxIter", "regParam", "rmse"]
)

display(comparison_df)

# Print the winner clearly
best_rmse = min(rmse_1, rmse_2, rmse_3)
best_name = ["Model 1", "Model 2", "Model 3"][[rmse_1, rmse_2, rmse_3].index(best_rmse)]
print(f"Best model: {best_name} with RMSE = {best_rmse:.4f}")

model,rank,maxIter,regParam,rmse
Model 1,10,10,0.1,115.88536796747813
Model 2,20,15,0.05,115.87811132248879
Model 3,5,10,0.5,115.89032696765078


Best model: Model 2 with RMSE = 115.8781


## Comparison of ALS Runs

The performance of the three ALS models was compared using RMSE, which measures the difference between predicted and actual interaction values. Lower RMSE values indicate better prediction accuracy.

Model 1 achieved an RMSE of 115.89, while Model 2 achieved a slightly lower RMSE of 115.88, making it the best-performing model. Model 3 achieved an RMSE of 115.89032696765078, which is slightly higher than Model 2 and very close to Model 1.

The results show that increasing the model complexity (as in Model 2) provides a small improvement in prediction accuracy. In contrast, reducing complexity and increasing regularisation (as in Model 3) slightly worsens performance. This demonstrates that model performance is sensitive to hyperparameter choices and highlights the importance of experimentation.

## Final Model Selection

After comparing the RMSE values of all three ALS models, Model 2 achieved the lowest RMSE (115.88) and was therefore selected as the final model.

Although the improvement over Model 1 is small, Model 2 still provides the most accurate predictions among the models tested. Model 3 performed slightly worse due to stronger regularisation and lower model complexity, which limited its ability to learn user preferences effectively.

Selecting Model 2 ensures that the recommender system is based on the most accurate configuration identified through experimentation. This model is therefore used to generate the final recommendations for users.

In [0]:
# Select the best model based on RMSE
if rmse_1 <= rmse_2 and rmse_1 <= rmse_3:
    best_model = model_1
    best_model_name = "Model 1"
elif rmse_2 <= rmse_1 and rmse_2 <= rmse_3:
    best_model = model_2
    best_model_name = "Model 2"
else:
    best_model = model_3
    best_model_name = "Model 3"

print(f"{best_model_name} was selected as the best model based on lowest RMSE.")

# Generate top 5 recommendations for all users
best_user_recommendations = best_model.recommendForAllUsers(5)

display(best_user_recommendations.limit(10))

Model 2 was selected as the best model based on lowest RMSE.


user_id,recommendations
76767,"List(List(2476, 1.3125399), List(982, 1.1981877), List(979, 1.1936826), List(737, 1.0133146), List(738, 0.9790988))"
86540,"List(List(3826, 1.3121897), List(4365, 0.9704023), List(3224, 0.95192754), List(2071, 0.8976708), List(3223, 0.8832628))"
144736,"List(List(979, 1.1581844), List(982, 1.0567585), List(980, 0.89324343), List(981, 0.70711315), List(1132, 0.68114346))"
229911,"List(List(985, 1.3177419), List(979, 1.1926868), List(2476, 1.184454), List(982, 1.1211808), List(737, 0.97755975))"
835015,"List(List(979, 0.8219423), List(982, 0.74992114), List(980, 0.6334186), List(981, 0.5020075), List(1132, 0.48467052))"
948368,"List(List(985, 0.9952795), List(2072, 0.75282234), List(979, 0.7418146), List(2075, 0.7341776), List(4365, 0.680703))"
975449,"List(List(3826, 1.2905167), List(2652, 1.1851559), List(2071, 1.1813536), List(3223, 1.1446534), List(2448, 1.1339737))"
976129,"List(List(979, 1.0195622), List(982, 0.930386), List(980, 0.7877557), List(981, 0.62341326), List(1132, 0.59939545))"
1024319,"List(List(979, 1.3226322), List(982, 1.2070419), List(980, 1.0211755), List(981, 0.8087454), List(1132, 0.7787172))"
1268792,"List(List(979, 1.0601897), List(982, 0.9673422), List(980, 0.81762135), List(981, 0.64723045), List(1132, 0.62346107))"


In [0]:
from pyspark.sql.functions import explode, col

# Make recommendations readable
best_rec_exploded = best_user_recommendations.select(
    "user_id",
    explode("recommendations").alias("rec")
)

best_rec_final = best_rec_exploded.select(
    "user_id",
    col("rec.game_id").alias("game_id"),
    col("rec.rating").alias("predicted_score")
)

best_rec_with_names = best_rec_final.join(games_df, on="game_id", how="inner")

display(best_rec_with_names.limit(20))

game_id,user_id,predicted_score,game
39,234467038,0.33242965,7 Days to Die
39,195171884,0.9470142,7 Days to Die
39,190675881,0.73805815,7 Days to Die
39,174287424,1.0649437,7 Days to Die
39,124651043,1.058278,7 Days to Die
39,105243827,1.1696684,7 Days to Die
39,101656933,1.0140504,7 Days to Die
39,101315169,0.030907525,7 Days to Die
39,92107940,0.8750895,7 Days to Die
39,46607365,1.1734408,7 Days to Die


### Explanation of Results

The final recommendations were generated using the better-performing ALS model. The results were then converted into a more readable format by mapping game IDs back to game names. This makes it possible to inspect the recommendations directly and understand which games are suggested to different users.

## Conclusion

In this task, a recommender system was developed using Spark MLlib and the ALS algorithm based on user interaction data from the Steam dataset. The dataset was first explored and preprocessed to create suitable user and game identifiers, as well as interaction values for modelling.

Three ALS models were trained using different hyperparameter configurations. This allowed a systematic comparison of model performance rather than relying on a single model. Each model was evaluated using RMSE, which was used to measure prediction accuracy.

The comparison showed that Model 2 achieved the lowest RMSE (115.88), making it the best-performing model. Although the improvement over Model 1 was small, it still demonstrated that hyperparameter tuning can influence model performance. Model 3 performed slightly worse, showing that stronger regularisation and reduced model complexity can lead to underfitting.

The selected model was then used to generate personalised game recommendations for users. These recommendations were transformed into a readable format by extracting predicted values and joining them with game information.

Overall, this task demonstrates how Spark can be used to process large-scale interaction data and build a collaborative filtering recommender system. It also highlights the importance of experimentation, evaluation, and evidence-based model selection in machine learning.

####Use of AI Tools

I have used AI tools to support understanding of the task. All work was reviewed, tested, and adapted independently within Databricks to ensure correctness and originality.